# Simplified Model and Constructive Heuristics

This notebook explores a **modified, relaxed** home healthcare routing model with synthetic inputs, a CBC solver, and three constructive heuristics (H1, H2, H3).

**Limitations:** The original time-sequencing constraint is omitted from the solved base model; the function for adding Lagrangian penalties is defined but is not invoked in the saved final execution (`pi=None`). Heuristic routes are initialized in the model but **not fixed** before the second solve. The final route printed by CBC therefore need not equal H2.


In [85]:

# 1) IMPORTS & BASIC SETUP

!apt-get install -y coinor-cbc  # or glpk-utils, if you prefer GLPK


from pyomo.environ import (
    ConcreteModel, Var, Objective, Constraint, RangeSet, Binary, NonNegativeReals,
    SolverFactory, value, minimize, Param
)
from pyomo.opt import SolverStatus, TerminationCondition
import math


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
coinor-cbc is already the newest version (2.10.7+ds1-1).
0 upgraded, 0 newly installed, 0 to remove and 18 not upgraded.


In [86]:
 #Base model (without constraint 9)

def create_base_model_relaxed():

    model = ConcreteModel()


    M = 3  #Number of patients
    N = 2  # Number of nurses
    K = 1  #Number of vehicles

    # Store values for use in other functions
    model.M = M
    model.N_ = N
    model.K_ = K


    Dij_data = [
        [0, 15, 15],  # Dist from patient1 to {1,2,3}
        [11, 0, 11],  # Dist from patient2 to {1,2,3}
        [9, 18, 0]   # Dist from patient3 to {1,2,3}

    ]

    # Ai: drugs needed, Bi: samples, Ei/Li: earliest & latest times, Wi: service time
    Ai_data = [3, 3, 3]     # all small demands
    Bi_data = [1, 2, 3]     # small sample
    Ei_data = [0, 0, 0]     # earliest time 0 (no real TW restriction)
    Li_data = [100, 100, 100] # large enough to not restrict feasibility
    Wi_data = [0, 0, 1]     # 1 unit service time each
    CAPk_data = [10]        # capacity is large enough
    TCk_data = [5]          # transport cost
    PEN = 1.5               # penalty factor
    BIG = 50                # smaller Big-M for a tiny example


    # Put them into Pyomo Param or store as attributes:
    model.Dij = Param(range(1, M+1), range(1, M+1),
                      initialize=lambda m,i,j: Dij_data[i-1][j-1],
                      within=NonNegativeReals)
    model.Ai = Param(range(1, M+1),
                     initialize=lambda m,i: Ai_data[i-1],
                     within=NonNegativeReals)
    model.Bi = Param(range(1, M+1),
                     initialize=lambda m,i: Bi_data[i-1],
                     within=NonNegativeReals)
    model.Ei = Param(range(1, M+1),
                     initialize=lambda m,i: Ei_data[i-1])
    model.Li = Param(range(1, M+1),
                     initialize=lambda m,i: Li_data[i-1])
    model.Wi = Param(range(1, M+1),
                     initialize=lambda m,i: Wi_data[i-1])
    model.CAPk = Param(range(1, K+1),
                       initialize=lambda m,k: CAPk_data[k-1])
    model.TCk = Param(range(1, K+1),
                       initialize=lambda m,k: TCk_data[k-1])
    model.PEN = PEN
    model.BIG = BIG


    model.P = RangeSet(1, M)      # patients
    model.Nurses = RangeSet(1, N) # nurses
    model.Kv = RangeSet(1, K)     # vehicles


    # Decision variables

    model.X = Var(model.Kv, model.P, model.P, model.Nurses, domain=Binary)
    model.S = Var(model.P, model.Nurses, domain=NonNegativeReals)
    model.O = Var(model.Nurses, model.Kv, domain=NonNegativeReals)

    # ----------------------
    # Base objective (no Lagrangian penalty for (9) yet)
    # Z = sum_{k,i,j,n} Dij[i,j]*TCk[k]*X[k,i,j,n] + sum_{n,k} O[n,k]*TCk[k]*PEN
    # ----------------------
    def base_obj_rule(m):
        routing_cost = sum(m.Dij[i,j]*m.TCk[k]*m.X[k,i,j,n]
                           for k in m.Kv for i in m.P for j in m.P for n in m.Nurses)
        penalty_cost = sum(m.O[n,k] * m.TCk[k] * m.PEN
                           for n in m.Nurses for k in m.Kv)
        return routing_cost + penalty_cost
    model.base_obj = Objective(rule=base_obj_rule, sense=minimize)

    # ----------------------
    # Constraints
    # (2) Each patient exactly once
    # (3) Route assignment
    # (4) Vehicle capacity (drugs)
    # (5) Vehicle capacity (samples)
    # (6) Start route? (In your problem you start from i=1 or from a depot)
    # (7) Flow conservation
    # (8) Ensure start route
    # (10) Earliest start
    # (11) Latest finish
    # (12) Distance traveled
    # We skip (9) here (time-sequencing).
    # ----------------------

    # (2) Each patient is visited exactly once
    def visit_once_rule(m, i):
      return sum(m.X[k, i, j, n] for k in m.Kv for j in m.P if j != i for n in m.Nurses) == 1
    model.VisitOnce = Constraint(model.P, rule=visit_once_rule)

   # def visit_once_rule(m, i):
       # return sum(m.X[k, i, j, n] for k in m.Kv for j in m.P for n in m.Nurses) == 1
   # model.VisitOnce = Constraint(model.P, rule=visit_once_rule)


    # (3) Route assignment
    def assignment_rule(m, i, j, n):
        return sum(m.X[k, i, j, n] for k in m.Kv) <= 1
    model.Assignment = Constraint(model.P, model.P, model.Nurses, rule=assignment_rule)

    # (4) Drug capacity
    def capacity_rule_drugs(m, k, n):
        return sum(m.Ai[i] * m.X[k, i, j, n] for i in m.P for j in m.P) <= m.CAPk[k]
    model.CapacityDrugs = Constraint(model.Kv, model.Nurses, rule=capacity_rule_drugs)

    # (5) Biological sample capacity
    def capacity_rule_samples(m, k, n):
        return sum(m.Bi[i] * m.X[k, i, j, n] for i in m.P for j in m.P) <= m.CAPk[k]
    model.CapacitySamples = Constraint(model.Kv, model.Nurses, rule=capacity_rule_samples)


    # (7) Flow balance
    def flow_conservation_rule(m, h, k, n):
        return sum(m.X[k, i, h, n] for i in m.P) == sum(m.X[k, h, j, n] for j in m.P)
    model.FlowConservation = Constraint(model.P, model.Kv, model.Nurses, rule=flow_conservation_rule)


    # (10) Earliest service start
    def time_window_start(m, i, n):
        return m.S[i, n] >= m.Ei[i]
    model.TimeWindowStart = Constraint(model.P, model.Nurses, rule=time_window_start)

    # (11) Latest service end
    def time_window_end(m, i, n):
        return m.S[i, n] <= m.Li[i]
    model.TimeWindowEnd = Constraint(model.P, model.Nurses, rule=time_window_end)

    # (12) Distance traveled
    def distance_calculation_rule(m, n, k):
        return m.O[n, k] == sum(m.Dij[i,j]*m.X[k, i, j, n] for i in m.P for j in m.P)
    model.DistanceCalculation = Constraint(model.Nurses, model.Kv, rule=distance_calculation_rule)

    return model

In [88]:
# 3) Lagrangian relaxation (optional)

def add_lagrangian_penalty(model, pi):
    """
   Constraint 9:
       S[j,n] >= S[i,n] + T[i,j] + W[i] - BIG*(1 - sum_k X[k,i,j,n])
    =>
       S[j,n] - S[i,n] - Dij[i,j] - Wi[i] + BIG*(1 - X[..]) >= 0

    penalty = sum_{i,j,n} pi_{i,j,n} * [LHS_of_(9)].
   """
    base_expr = model.base_obj.expr

    def penalty_expr(m):
        pen = 0
        for i in m.P:
            for j in m.P:
                for n in m.Nurses:
                    x_sum = sum(m.X[k, i, j, n] for k in m.Kv)
                    lhs = (m.S[j,n]
                           - m.S[i,n]
                           - m.Dij[i,j]
                           - m.Wi[i]
                           + m.BIG*(1 - x_sum))
                    pen += pi[(i,j,n)] * lhs
        return pen

    def relaxed_obj_rule(m):
        return base_expr + penalty_expr(m)

    model.del_component(model.base_obj)
    model.relaxed_obj = Objective(rule=relaxed_obj_rule, sense=minimize)

    return model

In [89]:
# 4) Solve the model with or without Lagrangian relaxation

def solve_relaxed_model(pi=None):
    """If pi is provided, add the penalty and solve; otherwise, solve the base objective.
    """
    m = create_base_model_relaxed()
    if pi is not None:
        add_lagrangian_penalty(m, pi)

    solver = SolverFactory('cbc')  # or 'glpk', 'gurobi', etc.
    results = solver.solve(m, tee=False, load_solutions=True)

    # Check whether the solver found a solution
    if (results.solver.status == SolverStatus.ok and
        results.solver.termination_condition == TerminationCondition.optimal):
        print("Model solved with an optimal solution.")
        if pi is None:
            print("Objective =", value(m.base_obj))
        else:
            print("Lagrangian-relaxed objective =", value(m.relaxed_obj))
    else:
        print("Solver ended with condition:", results.solver.termination_condition)
        print("No feasible/optimal solution found. Variables remain uninitialized.")

    return m, results

In [90]:
# 5) Heuristic algorithms (H1, H2, H3)

#   DTC[i][j] = D[i,j]*TC
# For convenience and simplicity, or just D[i,j]

def build_DTC_matrix(model):

    M = model.M
    TC = value(model.TCk[1])
    DTC = [[0]*(M+1) for _ in range(M+1)]
    for i in range(1, M+1):
        for j in range(1, M+1):
            DTC[i][j] = value(model.Dij[i,j]) * TC
    return DTC
#H1
def heuristic_H1(DTC, M, num_nurses):

    unvisited = set(range(1, M+1))
    routes = [[] for _ in range(num_nurses)]
    nurse_idx = 0

    while unvisited and nurse_idx < num_nurses:
        route = []
        current = 1
        if 1 in unvisited:
            route.append(1)
            unvisited.remove(1)
# Select the patient nearest to the first patient
        while unvisited:
            next_pt = None
            min_c = math.inf
            for j in unvisited:
                if DTC[current][j] < min_c:
                    min_c = DTC[current][j]
                    next_pt = j
            route.append(next_pt)
            unvisited.remove(next_pt)
            current = next_pt

            # Just break if we want only 1 route per nurse if that suits your problem
            # We'll keep going to fill up the route until we run out of unvisited in this simple example

        routes[nurse_idx] = route
        nurse_idx += 1

    return routes
#H2
def heuristic_H2(DTC, M, num_nurses):
  #Select the first patient
    unvisited = set(range(1, M+1))
    routes = [[] for _ in range(num_nurses)]
    nurse_idx = 0

    #  Compute the average distance for each patient i
    row_avg = {}
    for i in range(1, M+1):
        row_avg[i] = sum(DTC[i][j] for j in range(1, M+1) if j!=i)/(M-1)

    while unvisited and nurse_idx < num_nurses:
        # Select the lowest row average
        first_pt = min(unvisited, key=lambda x: row_avg[x])
        route = [first_pt]
        unvisited.remove(first_pt)
        current = first_pt

        # Select the nearest patient
        while unvisited:
            next_pt = None
            min_c = math.inf
            for j in unvisited:
                if DTC[current][j] < min_c:
                    min_c = DTC[current][j]
                    next_pt = j
            route.append(next_pt)
            unvisited.remove(next_pt)
            current = next_pt

        routes[nurse_idx] = route
        nurse_idx += 1

    return routes
#H3
def heuristic_H3(DTC, M, num_nurses):

    unvisited = set(range(1, M+1))
    routes = [[] for _ in range(num_nurses)]
    nurse_idx = 0

    # Start from the last patient
    last_patient = M

    while unvisited and nurse_idx < num_nurses:
        route = []
        # Select the last patient if still unvisited
        if last_patient in unvisited:
            route.append(last_patient)
            unvisited.remove(last_patient)
        current = last_patient

        # Nearest-neighbor patient
        while unvisited:
            next_pt = None
            min_c = math.inf
            for j in unvisited:
                if DTC[current][j] < min_c:
                    min_c = DTC[current][j]
                    next_pt = j
            route.append(next_pt)
            unvisited.remove(next_pt)
            current = next_pt

        routes[nurse_idx] = route
        nurse_idx += 1

        #Select the next route based on the last unvisited patient
        last_patient = current

    return routes


In [91]:
# 6) Apply the heuristic solution to the model

def apply_heuristic_solution_to_model(model, routes):

    k = 1
    # First, set all X=0
    for i in model.P:
        for j in model.P:
            for n in model.Nurses:
                model.X[k, i, j, n].value = 0

    nurse_idx = 1
    for route in routes:

        for idx in range(len(route)-1):
            i = route[idx]
            j = route[idx+1]
            #  Set X=1 for the nurses
            if i != j:
                model.X[k, i, j, nurse_idx].value = 1

        nurse_idx += 1
        if nurse_idx > model.N_:
            break


    for i in model.P:
        for n in model.Nurses:
            model.S[i,n].value = 0

    nurse_idx = 1
    for route in routes:
        current_time = 0
        for idx in range(len(route)):
            i = route[idx]
            model.S[i, nurse_idx].value = current_time
            if idx < len(route)-1:
                j = route[idx+1]
                travel_ij = value(model.Dij[i,j])
                service_i = value(model.Wi[i])
                current_time = current_time + travel_ij + service_i
        nurse_idx += 1
        if nurse_idx > model.N_:
            break

    return model

In [92]:
# 7) Final execution

if __name__ == "__main__":

    print("=== Step A: Solve the base model (no constraint(9) & no Lagrangian) ===")
    base_model, base_res = solve_relaxed_model(pi=None)


    print("\n=== Step B: Build cost matrix & run heuristics ===")
    DTC = build_DTC_matrix(base_model)  # 1-based DTC
    M = base_model.M
    num_nurses = base_model.N_

    routes_h1 = heuristic_H1(DTC, M, num_nurses)
    routes_h2 = heuristic_H2(DTC, M, num_nurses)
    routes_h3 = heuristic_H3(DTC, M, num_nurses)

    print("H1 routes:", routes_h1)
    print("H2 routes:", routes_h2)
    print("H3 routes:", routes_h3)

    chosen_routes = routes_h2
    print("\nApplying H2 routes into the model...")

    apply_heuristic_solution_to_model(base_model, chosen_routes)



    solver = SolverFactory('cbc')
    solver.options['mipstart'] = 1
    results = solver.solve(base_model, tee=False, load_solutions=True)

    if (results.solver.status == SolverStatus.ok and
        results.solver.termination_condition == TerminationCondition.optimal):
        print("After applying H2, solver found an optimal solution again (or the same).")
        print("Objective =", value(base_model.base_obj))
    else:
        print("Solver ended with", results.solver.termination_condition,
              "but we have a heuristic-based solution loaded in any case.")

    print("\n=== Final X & S values from H2-based solution ===")
    for n in base_model.Nurses:
        for i in base_model.P:
            for j in base_model.P:
                if value(base_model.X[1, i, j, n]) > 0.5:
                    print(f"  Nurse={n}: X[1,{i},{j},{n}] = 1")

    for i in base_model.P:
        for n in base_model.Nurses:
            s_val = value(base_model.S[i,n])
            if s_val is not None:
                print(f"  S[{i},{n}] = {s_val:.2f}")

=== Step A: Solve the base model (no constraint(9) & no Lagrangian) ===
Model solved with an optimal solution.
Objective = 437.5

=== Step B: Build cost matrix & run heuristics ===
H1 routes: [[1, 2, 3], []]
H2 routes: [[2, 1, 3], []]
H3 routes: [[3, 1, 2], []]

Applying H2 routes into the model...
After applying H2, solver found an optimal solution again (or the same).
Objective = 437.5

=== Final X & S values from H2-based solution ===
  Nurse=1: X[1,1,2,1] = 1
  Nurse=1: X[1,2,3,1] = 1
  Nurse=1: X[1,3,1,1] = 1
  S[1,1] = 0.00
  S[1,2] = 0.00
  S[2,1] = 0.00
  S[2,2] = 0.00
  S[3,1] = 0.00
  S[3,2] = 0.00
